# 01 Data check

Reads every dataset file in the project folder, lists every column with what it is used for (or why it is not used), and checks that the files agree with each other.

Reads: `herbs.tsv`, `compounds.tsv`, `herb_compound_edges.tsv`, `compound_target_labeled.csv`, `compound_target_unlabeled.csv`, `herb_compound_target.csv`, `compound_features_ecfp4.csv`, `target_features_aac.csv`, `target_features_esm2.csv`
Writes: nothing

## Setup

In [1]:
import hti
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from hti.data import DATA

RDLogger.DisableLog('rdApp.*')
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.max_rows', 200)

## Load all files
The unlabeled file has 3.1 million rows, so it is read with category columns to keep memory low.

In [2]:
herbs = pd.read_csv(DATA / 'herbs.tsv', sep='\t', dtype=str)
compounds = pd.read_csv(DATA / 'compounds.tsv', sep='\t', dtype=str)
herb_compound = pd.read_csv(DATA / 'herb_compound_edges.tsv', sep='\t', dtype=str)
labeled = pd.read_csv(DATA / 'compound_target_labeled.csv', dtype=str)
unlabeled = pd.read_csv(DATA / 'compound_target_unlabeled.csv', dtype='category')
herb_target = pd.read_csv(DATA / 'herb_compound_target.csv', dtype=str)
ecfp_file = pd.read_csv(DATA / 'compound_features_ecfp4.csv')
aac = pd.read_csv(DATA / 'target_features_aac.csv')
esm = pd.read_csv(DATA / 'target_features_esm2.csv')

files = {'herbs.tsv': herbs, 'compounds.tsv': compounds, 'herb_compound_edges.tsv': herb_compound,
         'compound_target_labeled.csv': labeled, 'compound_target_unlabeled.csv': unlabeled,
         'herb_compound_target.csv': herb_target, 'compound_features_ecfp4.csv': ecfp_file,
         'target_features_aac.csv': aac, 'target_features_esm2.csv': esm}
pd.DataFrame([{'file': name, 'rows': len(df), 'columns': df.shape[1]} for name, df in files.items()])

,file,rows,columns
0,herbs.tsv,62,13
1,compounds.tsv,2319,21
2,herb_compound_edges.tsv,8396,14
3,compound_target_labeled.csv,14166,15
4,compound_target_unlabeled.csv,3095613,3
5,herb_compound_target.csv,126830,22
6,compound_features_ecfp4.csv,2319,2049
7,target_features_aac.csv,1341,23
8,target_features_esm2.csv,1341,1282


## Every column and what happens to it
Columns are marked automatically as constant (same value in every row) or as a copy of an earlier column.
The rest are decided by hand below. Blocks of feature columns (`aac_*`, `esm_*`, `ecfp4_*`) are shown as one row.

In [3]:
use = {
    ('herbs.tsv', 'herb_id'): 'herb node id',
    ('herbs.tsv', 'botanical_name'): 'herb name in tables and predictions',
    ('herbs.tsv', 'Common_name'): 'herb name in predictions',
    ('herbs.tsv', 'Family'): 'herb feature: one-hot plant family (38 families)',
    ('compounds.tsv', 'compound_id'): 'compound node id',
    ('compounds.tsv', 'Phytochemical name_standardised'): 'compound name in predictions',
    ('compounds.tsv', 'Canonical_SMILES'): 'compound features: ECFP4, FCFP4 and MACCS fingerprints are computed from it',
    ('compounds.tsv', 'InChIKey'): 'first block groups stereoisomers, kept on one side of the new-compound split',
    ('herb_compound_edges.tsv', 'herb_id'): 'H-C links',
    ('herb_compound_edges.tsv', 'compound_id'): 'H-C links',
    ('compound_target_labeled.csv', 'compound_id'): 'C-T positives',
    ('compound_target_labeled.csv', 'protein_id'): 'C-T positives',
    ('compound_target_labeled.csv', 'hgnc_symbol'): 'target gene symbol in predictions',
    ('compound_target_unlabeled.csv', 'compound_id'): 'pool of unlabeled pairs, negatives are drawn from it',
    ('compound_target_unlabeled.csv', 'protein_id'): 'pool of unlabeled pairs, negatives are drawn from it',
    ('herb_compound_target.csv', 'herb_id'): 'checks that herb-target pairs equal the H-C x C-T join',
    ('herb_compound_target.csv', 'compound_id'): 'checks that herb-target pairs equal the H-C x C-T join',
    ('herb_compound_target.csv', 'protein_id'): 'checks that herb-target pairs equal the H-C x C-T join',
    ('compound_features_ecfp4.csv', 'compound_id'): 'checked only: this column is in the wrong order (see below)',
    ('compound_features_ecfp4.csv', 'ecfp4_*'): 'checked only: equal to the ECFP4 recomputed from SMILES, by row order',
    ('target_features_aac.csv', 'protein_id'): 'target node id',
    ('target_features_aac.csv', 'protein_name'): 'target name in predictions',
    ('target_features_aac.csv', 'aac_*'): 'target feature for the comparison model (20 amino acid shares)',
    ('target_features_esm2.csv', 'protein_id'): 'target node id',
    ('target_features_esm2.csv', 'esm_*'): 'main target feature (ESM-2, 1,280 numbers per protein)',
}
not_used = {
    ('herb_compound_edges.tsv', 'Plant_part'): 'not used: same values as plant_part',
    ('compound_target_labeled.csv', 'source'): 'not used: same values as evidence_source',
    ('herb_compound_edges.tsv', 'plant_part'): 'not used in the model: plant-part relations gave no gain in earlier tuning',
    ('compound_target_labeled.csv', 'evidence_source'): 'not used in the model: evidence weighting gave no gain in earlier tuning',
    ('compound_target_labeled.csv', 'n_supporting_records'): 'not used: 1 in every row except one',
    ('compounds.tsv', 'Scaffold Graph'): 'not used: missing for 359 acyclic compounds; stereoisomer grouping covers the leakage case',
}
blocks = {'aac_': 'aac_*', 'esm_': 'esm_*', 'ecfp4_': 'ecfp4_*'}

rows = []
for name, df in files.items():
    seen_blocks = set()
    for i, col in enumerate(df.columns):
        prefix = next((p for p in blocks if col.startswith(p)), None)
        if prefix:
            if prefix in seen_blocks:
                continue
            seen_blocks.add(prefix)
            n_cols = sum(c.startswith(prefix) for c in df.columns)
            rows.append({'file': name, 'column': f'{blocks[prefix]} ({n_cols} columns)', 'missing': int(df.filter(like=prefix).isna().sum().sum()),
                         'distinct': '', 'decision': use.get((name, blocks[prefix]), '')})
            continue
        values = df[col].astype(str)
        copy_of = next((c for c in df.columns[:i] if not any(c.startswith(p) for p in blocks) and values.equals(df[c].astype(str))), None)
        if (name, col) in use:
            decision = use[(name, col)]
        elif (name, col) in not_used:
            decision = not_used[(name, col)]
        elif copy_of:
            decision = f'not used: same values as {copy_of}'
        elif df[col].nunique(dropna=False) == 1:
            decision = f'not used: same value in every row ({df[col].iloc[0]})'
        else:
            decision = 'not used: identifier, link or free text from another database, no signal for prediction'
        rows.append({'file': name, 'column': col, 'missing': int(df[col].isna().sum()),
                     'distinct': int(df[col].nunique()), 'decision': decision})
inventory = pd.DataFrame(rows)
inventory

,file,column,missing,distinct,decision
0,herbs.tsv,herb_id,0,62,herb node id
1,herbs.tsv,herb_source_id,0,62,"not used: identifier, link or free text from another database, no signal for prediction"
2,herbs.tsv,botanical_name,0,62,herb name in tables and predictions
3,herbs.tsv,botanical_name_normalized,0,62,"not used: identifier, link or free text from another database, no signal for prediction"
4,herbs.tsv,Plant_identifier,0,62,not used: same values as herb_source_id
5,herbs.tsv,Indian_Medicinal_plant,0,62,not used: same values as botanical_name
6,herbs.tsv,Synonymous names,0,62,"not used: identifier, link or free text from another database, no signal for prediction"
7,herbs.tsv,Kingdom,0,1,not used: same value in every row (Plantae)
8,herbs.tsv,Family,0,38,herb feature: one-hot plant family (38 families)
9,herbs.tsv,Group,0,1,not used: same value in every row (Angiosperms)


## Ids and links agree across files

In [4]:
herb_set = set(herbs['herb_id'])
compound_set = set(compounds['compound_id'])
target_set = set(aac['protein_id'])
hc_pairs = herb_compound[['herb_id', 'compound_id']].drop_duplicates()
ct_pairs = labeled[['compound_id', 'protein_id']]

checks = {
    'herb ids unique': herbs['herb_id'].is_unique,
    'compound ids unique': compounds['compound_id'].is_unique,
    'target ids unique': aac['protein_id'].is_unique,
    'fingerprint file has the same compounds': set(ecfp_file['compound_id']) == compound_set,
    'ESM-2 file has the same targets': set(esm['protein_id']) == target_set,
    'protein_id = UNIPROT:<accession>': (aac['protein_id'] == 'UNIPROT:' + aac['uniprot_accession']).all(),
    'H-C links use known herbs': set(hc_pairs['herb_id']) <= herb_set,
    'H-C links use known compounds': set(hc_pairs['compound_id']) <= compound_set,
    'every compound is in some herb': set(hc_pairs['compound_id']) == compound_set,
    'C-T links use known compounds': set(ct_pairs['compound_id']) <= compound_set,
    'C-T links use known targets': set(ct_pairs['protein_id']) <= target_set,
    'every target has a known compound': set(ct_pairs['protein_id']) == target_set,
    'no duplicate C-T links': not ct_pairs.duplicated().any(),
    'every label is 1': (labeled['label'] == '1').all(),
}
pd.Series(checks, name='ok').to_frame()

,ok
herb ids unique,True
compound ids unique,True
target ids unique,True
fingerprint file has the same compounds,True
ESM-2 file has the same targets,True
protein_id = UNIPROT:<accession>,True
H-C links use known herbs,True
H-C links use known compounds,True
every compound is in some herb,True
C-T links use known compounds,True


## The 3.1 million unlabeled pairs
They should be exactly every compound x target pair that is not a known link: no duplicates, no overlap with the known links.

In [5]:
n_c, n_t = len(compound_set), len(target_set)
c_index = {c: i for i, c in enumerate(sorted(compound_set))}
t_index = {t: i for i, t in enumerate(sorted(target_set))}

def codes(col, index):
    lookup = np.array([index[x] for x in col.cat.categories])
    return lookup[col.cat.codes.to_numpy()]

unlabeled_key = codes(unlabeled['compound_id'], c_index) * n_t + codes(unlabeled['protein_id'], t_index)
positive_key = ct_pairs['compound_id'].map(c_index).to_numpy() * n_t + ct_pairs['protein_id'].map(t_index).to_numpy()
complement = np.setdiff1d(np.arange(n_c * n_t), positive_key)

print('unlabeled pairs           ', len(unlabeled_key))
print('known pairs               ', len(positive_key))
print('all compound x target     ', n_c * n_t)
print('no duplicates             ', len(np.unique(unlabeled_key)) == len(unlabeled_key))
print('no overlap with known     ', not np.isin(unlabeled_key, positive_key).any())
print('equals every other pair   ', np.array_equal(np.sort(unlabeled_key), complement))
print('label_status values       ', unlabeled['label_status'].value_counts().to_dict())

unlabeled pairs            3095613
known pairs                14166
all compound x target      3109779
no duplicates              True
no overlap with known      True
equals every other pair    True
label_status values        {'UNLABELED': 3095613}


## Herb-target pairs
`herb_compound_target.csv` should hold exactly the herb-target pairs you get by joining H-C links with C-T links.
Herb-target pairs are not separate evidence, so in the models they are always computed from compounds.

In [6]:
joined = hc_pairs.merge(ct_pairs, on='compound_id')
from_join = set(zip(joined['herb_id'], joined['protein_id']))
from_file = set(zip(herb_target['herb_id'], herb_target['protein_id']))
print('herb-target pairs in file   ', len(from_file))
print('herb-target pairs from join ', len(from_join))
print('identical                   ', from_file == from_join)
print('every file row is a real H-C link + C-T link',
      herb_target.merge(hc_pairs).merge(ct_pairs).shape[0] == len(herb_target))

herb-target pairs in file    24835
herb-target pairs from join  24835
identical                    True
every file row is a real H-C link + C-T link True


## Fingerprint file
ECFP4 (Morgan radius 2, 2,048 bits) is recomputed from every compound's own SMILES with RDKit and compared with the file,
once by the file's `compound_id` column and once by row position in sorted id order.

In [7]:
smiles = compounds.set_index('compound_id')['Canonical_SMILES']
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
ids_sorted = sorted(compound_set)
recomputed = np.stack([morgan.GetFingerprintAsNumPy(Chem.MolFromSmiles(smiles[c])) for c in ids_sorted]).astype(np.uint8)
file_bits = ecfp_file.drop(columns='compound_id').to_numpy().astype(np.uint8)
row_of = {c: i for i, c in enumerate(ids_sorted)}
by_id_column = recomputed[[row_of[c] for c in ecfp_file['compound_id']]]

print('rows whose id is not the sorted id    ', int((ecfp_file['compound_id'].to_numpy() != np.array(ids_sorted)).sum()), 'of', len(ids_sorted))
print('identical rows, file id column       ', int((by_id_column == file_bits).all(1).sum()))
print('identical rows, sorted row position  ', int((recomputed == file_bits).all(1).sum()))
print('so the bit rows are right, the id column is wrong; the pipeline computes fingerprints from SMILES')

rows whose id is not the sorted id     2317 of 2319
identical rows, file id column        2
identical rows, sorted row position   2319
so the bit rows are right, the id column is wrong; the pipeline computes fingerprints from SMILES


## Compounds that look the same to the models
Compounds with an identical fingerprint, or with the same InChIKey first block (same structure, different stereo form),
would leak answers if one were in training and the other in the test set of the new-compound split. 02_prep keeps each group together.

In [8]:
same_bits = pd.Series([r.tobytes() for r in recomputed]).duplicated(keep=False)
skeleton = compounds.set_index('compound_id').loc[ids_sorted, 'InChIKey'].str[:14]
print('compounds sharing a fingerprint     ', int(same_bits.sum()))
print('compounds sharing an InChIKey block ', int(skeleton.duplicated(keep=False).sum()))
print('targets with an identical ESM-2 row ', int(esm.filter(like='esm_').duplicated(keep=False).sum()))
print('targets with an identical AAC row   ', int(aac.filter(like='aac_').round(12).duplicated(keep=False).sum()))

compounds sharing a fingerprint      470
compounds sharing an InChIKey block  318
targets with an identical ESM-2 row  0
targets with an identical AAC row    0


## Summary

In [9]:
pd.Series({
    'herbs': len(herb_set),
    'plant families': herbs['Family'].nunique(),
    'compounds': len(compound_set),
    'compounds with a known target': ct_pairs['compound_id'].nunique(),
    'targets': len(target_set),
    'H-C links (unique)': len(hc_pairs),
    'C-T known links': len(ct_pairs),
    'C-T unlabeled pairs': len(unlabeled_key),
    'herb-target pairs (derived)': len(from_join),
    'herb x compound pairs without a link': len(herb_set) * len(compound_set) - len(hc_pairs),
}, name='count').to_frame()

,count
herbs,62
plant families,38
compounds,2319
compounds with a known target,726
targets,1341
H-C links (unique),5005
C-T known links,14166
C-T unlabeled pairs,3095613
herb-target pairs (derived),24835
herb x compound pairs without a link,138773
